In [1]:
import base64
import io
import pickle
import functools
import itertools

import jax.numpy as jnp
import jax
jax.config.update("jax_platforms", "cpu")
jax.config.update("jax_debug_key_reuse", True)

# set cache size to 1GB
jax.config.update("jax_compilation_cache_max_size", 2**30 - 1)

# jax.config.update("jax_log_compiles", True)
# jax.config.update("jax_compiler_detailed_logging_min_ops", 50)

# jax.config.update("jax_explain_cache_misses", True)
# jax.config.update("jax_check_tracer_leaks", True)
# jax.config.update("jax_debug_nans", True)

import orthax

import matplotlib
import matplotlib.pyplot as plt
# import matplotlib.animation as animation
import pandas as pd
from scipy.io import loadmat, savemat
from scipy.stats import ranksums

# import plotly.graph_objects as go

from hnmf_tr_optimizer.hnmf_optimizer import HNMFOptimizer, NewHNMFOptimizer, PerturbanceHNMFOptimizer, RedoHNMFOptimizer
from hnmf_tr_optimizer.clusts import result_analysis

from dls_model import InitParamsGenerator2, clustering_preprocess
# plt.style.use('Solarize_Light2')

# from IPython.display import Markdown, display



ERROR:2025-08-25 18:03:45,498:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/gfertt/.local/lib/python3.10/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 304; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


## models, optimizers, helpers, etc

In [2]:
# Plotting theme setup


TEXT_COLOR = "white"
BG_COLOR = "black"

plt.rcParams["axes.facecolor"] = plt.rcParams["figure.facecolor"] = BG_COLOR
plt.rcParams["text.color"] = TEXT_COLOR
plt.rcParams["axes.labelcolor"] = TEXT_COLOR
plt.rcParams["xtick.color"] = TEXT_COLOR
plt.rcParams["ytick.color"] = TEXT_COLOR
plt.rcParams.update({
	"axes.grid" : True,
	"grid.color": "green",
	"grid.alpha": 0.35,
	"grid.linestyle": (0, (10, 10)),
})

# BETTER SIZES
DEFAULT_W, DEFAULT_H = (16, 9)
plt.rcParams["figure.figsize"] = [DEFAULT_W, DEFAULT_H]
plt.rcParams["font.size"] = 14
plt.rcParams["figure.dpi"] = 90

plt.style.use('dark_background')


In [ ]:
def scatter_vector(theta, lambda_0=633e-9, n=1.33, radians=False):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    if not radians:
        theta = jnp.radians(theta)
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=1.38e-23, T=298.15, eta=0.00089):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def prep_data(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1

    g1 = jnp.where(
        jnp.greater_equal(g1_squared, 0),
        jnp.sqrt(g1_squared),
        -jnp.sqrt(-g1_squared)
    )

    # g1 = jnp.sign(g1_squared) * jnp.sqrt(jnp.abs(g1_squared))

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g1

def prep_data_g2(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    # div = jax.vmap(lambda x: x/x[0])
    # g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    # g1 = jnp.where(
    #     jnp.greater_equal(g1_squared, 0),
    #     jnp.sqrt(g1_squared),
    #     -jnp.sqrt(-g1_squared)
    # )

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g2_minus1_obs

# for drawing normal curves
def normal_distribution_single(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)

normal_distributions = jax.vmap(normal_distribution_single, in_axes=(None, 0, 0, 0))

def normal_distribution(possible_D, amp, mu, sig):
    whole = normal_distributions(possible_D, amp, mu, sig).sum(axis=0)
    return whole / jnp.sum(whole) # normalize for plotting


In [4]:
### models

SCALING_CONST = 2.45e-7


######### Normal model ##########

def get_g1(t, nk, a, c):
    b = a**2/2
    d = jnp.sqrt(2)
    s_pi = jnp.sqrt(jnp.pi)
    x = (-a + c*t)/d
    y = jnp.where(
        jnp.greater_equal(x, 5),
        1/(x*s_pi),
        jax.scipy.special.erfc(x)*jnp.exp(x**2)
    )
    e = (nk/2)*jnp.exp(-b)
    return e*y

# vectorize along time dimension
all_g1 = jax.vmap(
    get_g1,
    in_axes=(0, None, None, None)
)

def source3_1(t, q, amp, mu, sig):
    ################
    const = SCALING_CONST
    # const = 1.0
    ################
    nk = amp
    sig = sig*const
    mu = mu*const
    a = mu/sig
    c = q**2*sig

    g = all_g1(t, nk, a, c)
    return g

by_Xs1 = jax.vmap(
    source3_1,
    in_axes=(None, None, 0, 0, 0)
)

source_matrix1 = jax.vmap(
    by_Xs1,
    in_axes=(None, 0, None, None, None)
)

def g1_matrix(q, t, amp, mu, sig):
    full = source_matrix1(t, q, amp, mu, sig)
    full = jnp.sum(full, axis=1)
    return full

def g2_minus1_matrix(q, t, amp, mu, sig, beta):
    g1 = g1_matrix(q, t, amp, mu, sig)
    g2_minus1 = beta * g1**2
    return g2_minus1

######### Dirac model ##########

def single_exp(q, t, D, amp):
    return amp * jnp.exp(-D * q**2 * t)

by_Xs = jax.vmap(
    single_exp,
    in_axes=(None, None, 0, 0)
)

by_t = jax.vmap(
    by_Xs,
    in_axes=(None, 0, None, None)
)

by_q = jax.vmap(
    by_t,
    in_axes=(0, None, None, None)
)

def g1_dirac(q, t, D, amp, const):
    D_ = D * const
    full = by_q(q, t, D_, amp)
    full = jnp.sum(full, axis=2)
    return full

def g2_minus1_matrix_dirac(q, t, D, amp, beta, const):
    amp = amp / jnp.sum(amp)
    g1 = g1_dirac(q, t, D, amp, const)
    g2_minus1 = beta * g1**2
    return g2_minus1


def gen_bounds_dirac(k):
    return (1e-9*jnp.ones(k),1e-9*jnp.ones(k),jnp.array([0.0])), (1e-3*jnp.ones(k), jnp.ones(k),jnp.array([1.0]))



In [5]:
### Optimizers
min_k = 1
max_k = 2


# small_particle_bound = 1e-10 # 1 angstrom
# large_particle_bound = 1e-5 # 10 microns
def gen_bounds_std(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        jnp.array([1e-9])
    )
    upper_bounds = (
        1e-3*jnp.ones(num_sources),
        jnp.array([1.0])
    )
    return lower_bounds, upper_bounds

def gen_bounds_std_g1(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
    )
    upper_bounds = (
        1e-3*jnp.ones(num_sources),
    )
    return lower_bounds, upper_bounds


def gen_bounds_normal_g1(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
    )
    upper_bounds = (
        jnp.inf*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
    )
    return lower_bounds, upper_bounds

def extract_point(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    D = sol[0]
    amp = sol[1]
    beta = sol[2]
    if isinstance(amp, float):
        D = jnp.array([D])
        amp = jnp.array([amp])
        beta = jnp.array([beta])
    else:
        D = jnp.array(D)
        amp = jnp.array(amp)
        beta = jnp.array(beta)
    amp = amp/jnp.sum(amp)
    for p in range(len(amp)):
        point = jnp.stack([D[p], amp[p], beta[p]]).tolist()
        points.append(point)
    return points

def clustering_preprocess(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point, axis=1)
    return res

def filter_quantile(res, col_to_filter='fval', quantile=0.25):
    mod_col = res[col_to_filter].apply(lambda x: jnp.inf if jnp.isnan(x) else x)
    return res[
        mod_col < mod_col.quantile(q=quantile)
    ]

def process_res_dirac_(all_res, obs_size):
    Forclusts = clustering_preprocess(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        # group['normF'].mean(),
        group['fval'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts



def extract_point_std(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    sig = sol[0]
    beta = sol[1]
    beta = jnp.array(beta)
    if isinstance(sig, float):
        sig = jnp.array([sig])
    else:
        sig = jnp.array(sig)
    for p in range(len(sig)):
        point = sig.reshape(-1, 1)[p].tolist()
        points.append(point)
    return points

def clustering_preprocess_std(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point_std, axis=1)
    return res

def process_res_std_(all_res, obs_size):
    Forclusts = clustering_preprocess_std(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        # group['normF'].mean(),
        group['fval'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts

def extract_point_std_g1(row):
    # extract source amplitudes and positions to use as points for clustering
    points = []
    sol = row['sol']
    sig = sol[0]
    if isinstance(sig, float):
        sig = jnp.array([sig])
    else:
        sig = jnp.array(sig)
    for p in range(len(sig)):
        point = sig.reshape(-1, 1)[p].tolist()
        points.append(point)
    return points

def clustering_preprocess_std_g1(res):
    res = res.copy().groupby('num_sources', group_keys=False)[res.columns.tolist()].apply(filter_quantile)
    res['points'] = res.apply(extract_point_std_g1, axis=1)
    return res

def process_res_std_g1_(all_res, obs_size):
    Forclusts = clustering_preprocess_std_g1(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        # group['normF'].mean(),
        group['fval'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts

from dls_model import clustering_preprocess as clustering_preprocess_std_normal

def process_res_normal_(all_res, obs_size):
    Forclusts = clustering_preprocess_std_normal(all_res)
    Forclusts = Forclusts.groupby('num_sources', group_keys=False)[Forclusts.columns.tolist()].apply(lambda group: result_analysis(
        group['points'].sum(),
        # group['normF'].mean(),
        group['fval'].mean(),
        obs_size,
        group['num_sources'].iloc[0]
    ))
    Forclusts = Forclusts.set_index('num_source')

    return Forclusts

def l_statistic(full_sols, clust_info, sill_threshold=0.6, p_threshold=0.05):
    p_values = {}
    errors = {}
    n_opt = 1
    for k in clust_info[clust_info['min_sillhouette_score'] > sill_threshold].index:
        current_errors = full_sols[full_sols['num_sources'] == k]['normF'].sort_values().to_list()
        errors[k] = current_errors

        # For the second valid k onwards, perform the statistical test
        if k > 1:
            # Get the errors from the previous valid k
            prev_errors = errors[k-1]
            
            # Wilcoxon rank-sum test to see if the new errors are significantly smaller
            # We use a one-sided test ('less') to check if the current error distribution
            # is stochastically less than the previous one.
            _, p_val = ranksums(current_errors, prev_errors, alternative='less')
            p_values[k] = p_val
            
            # If the result is significant, this k is a better model
            if p_val < p_threshold:
                n_opt = k
    return n_opt, p_values, errors

def l_statistic2(full_sols, clust_info, observations, q, t, sill_threshold=0.6, p_threshold=0.05):
    p_values = {}
    errors = {}
    n_opt = 1
    for k in clust_info[clust_info['min_sillhouette_score'] > sill_threshold].index:
        best_amp, best_D, best_sig, best_beta = full_sols[full_sols['num_sources'] == k].sort_values('fval').iloc[0]['sol']
        recon = g2_minus1_matrix(q, t, best_amp, best_D, best_sig, best_beta)
        current_errors = jnp.zeros(len(q))
        # current_errors = []
        for qs in range(len(q)):
            s_segment = observations[qs, :]
            shat_segment = recon[qs, :]
            
            # Calculate the relative vector norm (error)
            error_norm = jnp.linalg.norm(shat_segment - s_segment)
            signal_norm = jnp.linalg.norm(s_segment)
            # Avoid division by zero if a signal segment is all zeros
            current_errors = current_errors.at[qs].set(error_norm / signal_norm if signal_norm > 0 else 0)

        errors[k] = current_errors


        # For the second valid k onwards, perform the statistical test
        if k > 1:
            # Get the errors from the previous valid k
            prev_errors = errors[k-1]
            
            # Wilcoxon rank-sum test to see if the new errors are significantly smaller
            # We use a one-sided test ('less') to check if the current error distribution
            # is stochastically less than the previous one.
            _, p_val = ranksums(current_errors, prev_errors, alternative='less')
            p_values[k] = p_val
            
            # If the result is significant, this k is a better model
            if p_val < p_threshold:
                n_opt = k
    return n_opt, p_values, errors

## run stuff

In [6]:
# import some experimental data
q, t, observations_100 = prep_data("~/repos/DLS/Experimental_data_083122/stock_100nm.csv")
q, t, observations_200 = prep_data("~/repos/DLS/Experimental_data_083122/stock_200nm.csv")
q, t, observations_500 = prep_data("~/repos/DLS/Experimental_data_083122/stock_500nm.csv")
q, t, observations_1000 = prep_data("~/repos/DLS/Experimental_data_083122/stock_1000nm.csv")
q, t, mix_1 = prep_data("~/repos/DLS/Experimental_data_083122/mix_1.csv")
q, t, mix_2 = prep_data("~/repos/DLS/Experimental_data_083122/mix_2.csv")
q, t, mix_3 = prep_data("~/repos/DLS/Experimental_data_083122/mix_3.csv")
q, t, mix_4 = prep_data("~/repos/DLS/Experimental_data_083122/mix_4.csv")

q, t, observations_100_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_100nm.csv")
q, t, observations_200_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_200nm.csv")
q, t, observations_500_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_500nm.csv")
q, t, observations_1000_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/stock_1000nm.csv")
q, t, mix_2_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_2.csv")
q, t, mix_1_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_1.csv")
q, t, mix_3_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_3.csv")
q, t, mix_4_g2 = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_4.csv")


mix_avg_g2 = jnp.mean(jnp.stack([mix_1_g2, mix_2_g2, mix_3_g2, mix_4_g2]), axis=0)


process_res_dirac = functools.partial(process_res_dirac_, obs_size=mix_1.size)
process_res_std = functools.partial(process_res_std_, obs_size=mix_1.size)
process_res_std_g1 = functools.partial(process_res_std_g1_, obs_size=mix_1.size)
process_res_normal = functools.partial(process_res_normal_, obs_size=mix_1.size)

In [7]:
def generate_distributions_by_distance(amp_pairs, lower_means, mean_distances, std_devs):
    core_params = list(itertools.product(std_devs, lower_means, mean_distances))
    all_combinations = []
    for amp1, amp2 in amp_pairs:
        for std, mean1, dist in core_params:
            mean2 = mean1 + dist
            all_combinations.append([amp1, mean1, std, amp2, mean2, std])

    return jnp.array(all_combinations)
    # all_combinations_jnp = jnp.array(all_combinations)
    # _amp1, mean1, std1, _amp2, mean2, std2 = all_combinations_jnp.T
    # a1 = (mean1 / 10) <= std1
    # a2 = (mean1 / 2.5) >= std1
    # a3 = (mean2 / 10) <= std2
    # a4 = (mean2 / 2.5) >= std2
    # a5 = (mean1 != mean2)

    # valid_mask = jnp.all(
    #     jnp.array([
    #         a1, a2, a3, a4, a5
    #     ]),
    #     axis=0
    # )

    # return all_combinations_jnp[valid_mask]


amplitude_pairs = [[0.5, 0.5]]
means = jnp.arange(1.8e-6, 1.9e-5, 6e-6).tolist()
mean_diffs = jnp.arange(5e-7, 5.1e-6, 1.5e-6).tolist()
stds = jnp.arange(6.5e-7, 1.31e-6, 3e-7).tolist()

distanced_params = generate_distributions_by_distance(amplitude_pairs, means, mean_diffs, stds)
amp1, mean1, std1, amp2, mean2, std2 = distanced_params.T
valid_params = distanced_params = jnp.stack([amp1, amp2, mean1, mean2, std1, std2]).T.reshape(distanced_params.shape[0], 3, 2)


In [9]:
amp, mu, sig = valid_params[10]
beta = 0.7

In [10]:
g2_minus1_matrix(q, t, amp, mu, sig, beta)

Array([[6.99950187e-001, 6.99925281e-001, 6.99900377e-001, ...,
        2.94233639e-078, 8.49620398e-084, 3.32428926e-089],
       [6.99932759e-001, 6.99899141e-001, 6.99865525e-001, ...,
        1.25300315e-100, 2.51145326e-107, 8.75746831e-114],
       [6.99913015e-001, 6.99869526e-001, 6.99826040e-001, ...,
        3.66737039e-123, 1.10141059e-130, 8.35739678e-138],
       ...,
       [6.99343648e-001, 6.99015705e-001, 6.98687917e-001, ...,
        2.98405871e-201, 2.34910168e-201, 1.89717274e-201],
       [6.99323920e-001, 6.98986127e-001, 6.98648499e-001, ...,
        2.72811695e-201, 2.15494794e-201, 1.74511170e-201],
       [6.99306509e-001, 6.98960023e-001, 6.98613710e-001, ...,
        2.52861976e-201, 2.00290849e-201, 1.62559201e-201]],      dtype=float64)

In [7]:
[SCALING_CONST * std for std in stds]
# widths

[1.5925e-13, 2.3275e-13, 3.0624999999999993e-13]

In [8]:
[SCALING_CONST * mean for mean in means]
# lower means

[4.4099999999999994e-13, 1.911e-12, 3.381e-12]

In [9]:
[SCALING_CONST * mean_diff for mean_diff in mean_diffs]
# mean diff

[1.2249999999999998e-13,
 4.899999999999999e-13,
 8.574999999999999e-13,
 1.225e-12]

In [10]:
# simulating shot (poisson) noise
def add_poisson_noise(g2_ideal, rand_key, average_counts_khz=1500, baseline=1.0):
    """
    g2_ideal - ideal g2 function
    average_counts_khz - average counts per channel in kHz
    seed - random seed for reproducibility
    """
    rand_key, subkey = jax.random.split(rand_key)
    mean_counts_per_channel = average_counts_khz * 100 # A proxy for total photon budget
    # The mean number of photons at each delay time τ is proportional to the ideal g2(τ)
    mean_photons_at_tau = mean_counts_per_channel * g2_ideal
    # Generate the noisy g2 data by drawing from a Poisson distribution
    # for each channel. This is the core of the shot noise simulation. 🎲
    noisy_counts = jax.random.poisson(subkey, mean_photons_at_tau)

    # Normalize the noisy counts to get the final noisy g2 function
    # The baseline of the noisy data is the average of the counts at long delay times
    noisy_baseline = jnp.mean(noisy_counts[:, -20:], axis=1) # Use last 20 channels for baseline
    g2_noisy = jax.vmap(lambda x, y: x/y)(noisy_counts, noisy_baseline)
    noisy_g2_minus_1 = g2_noisy - baseline  # Adjust the baseline to match the ideal g2

    return noisy_g2_minus_1, rand_key

def simulate_noisy_g2(
    g2_ideal: jnp.ndarray,
    rand_key: jax.random.PRNGKey,
    count_rate_khz: float,
    duration_s: float,
    baseline: float = 1.0,
    noise_scaling_factor: float = 0.1
) -> tuple[jnp.ndarray, jax.random.PRNGKey]:
    """
    Adds realistic Poisson noise to an ideal g2 autocorrelation function.

    This function simulates the shot noise inherent in a DLS experiment based on
    the instrument's count rate and the total measurement time.

    Args:
        g2_ideal: The ideal, noiseless g2 function (should not have the baseline subtracted).
                  Shape should be (batch, num_channels).
        rand_key: JAX random key for reproducibility.
        count_rate_khz: The average photon count rate in kHz (e.g., 20-45 from the manual).
        duration_s: The total duration of the experiment in seconds (e.g., 10, 30, 60).
        baseline: The theoretical baseline of the correlation function (typically 1.0).
        noise_scaling_factor: An empirical factor to match simulation to a real
                              correlator's output. It bridges the gap between total
                              photons and the statistical quality of the g2 function.
                              A value between 0.05 and 0.2 is a good starting point.

    Returns:
        A tuple containing:
        - noisy_g2_minus_1: The noisy g2 function with the baseline subtracted.
        - rand_key: The updated JAX random key.
    """
    # 1. Calculate the effective number of photon counts that contribute to the
    #    baseline of the correlation function. This is our "photon budget" and
    #    is the primary determinant of the noise level. It combines the
    #    instantaneous rate with the total measurement time.
    #    Total photons = count_rate_khz * 1000 * duration_s.
    #    The noise_scaling_factor adjusts this to better match the statistics
    #    of a real hardware correlator's averaging process.
    mean_counts_at_baseline = (
        count_rate_khz * 1000 * duration_s * noise_scaling_factor
    )

    # 2. The mean number of photons at each delay time τ is proportional to the ideal g2(τ).
    #    This creates the shape of the correlation function.
    mean_photons_at_tau = mean_counts_at_baseline * g2_ideal

    # 3. Generate the noisy data by drawing from a Poisson distribution for each channel.
    #    This is the core of the shot noise simulation.
    rand_key, subkey = jax.random.split(rand_key)
    noisy_counts = jax.random.poisson(subkey, mean_photons_at_tau)

    # 4. Normalize the noisy counts to get the final noisy g2 function.
    #    A robust method for finding the baseline of the noisy data is to average
    #    the counts from the last ~10% of the channels, where the function has decayed.
    num_channels_for_baseline = noisy_counts.shape[-1] // 10
    noisy_baseline = jnp.mean(noisy_counts[..., -num_channels_for_baseline:], axis=-1, keepdims=True)

    # Avoid division by zero if the baseline is somehow zero
    noisy_baseline = jnp.where(noisy_baseline == 0, 1.0, noisy_baseline)

    g2_noisy = noisy_counts / noisy_baseline

    # 5. Adjust by the theoretical baseline to center the result around 0.
    noisy_g2_minus_1 = g2_noisy - baseline

    return noisy_g2_minus_1, rand_key


def gen_data(
        q,
        t,
        valid_params,
        ensemble_size,
        gen_beta=0.7,
        baseline=1.0,
        rand_key=jax.random.key(1337),
        count_rate_khz=45.0,
        experiment_measurement_time_s=30.0,
        noise_scaling_factor=0.1
    ):
    clean_obs_list = []
    noisy_obs_list = []
    noise_errors = []
    snrs = []
    for i in range(valid_params.shape[0]):
        amp, mu, sig = valid_params[i]
        clean_g1 = g1_matrix(q, t, amp, mu, sig)
        g2_ideal = baseline + gen_beta*(clean_g1**2)


        # ensemble_observations = []
        # for _ in range(ensemble_size):
        #     noisy_g2_minus_1, rand_key = add_poisson_noise(g2_ideal, rand_key, average_counts_khz=average_counts_khz, baseline=baseline)
        #     ensemble_observations.append(noisy_g2_minus_1)
        # noisy_g2_minus_1 = jnp.mean(jnp.array(ensemble_observations), axis=0)

        # noisy_g2_minus_1, rand_key = add_poisson_noise(g2_ideal, rand_key, average_counts_khz=average_counts_khz, baseline=baseline)

        ensemble_observations = []
        for _ in range(ensemble_size):
            noisy_g2_minus_1, rand_key = simulate_noisy_g2(
                g2_ideal,
                rand_key,
                count_rate_khz=count_rate_khz,
                duration_s=experiment_measurement_time_s,
                baseline=baseline,
                noise_scaling_factor=noise_scaling_factor
            )
            ensemble_observations.append(noisy_g2_minus_1)
        noisy_g2_minus_1 = jnp.mean(jnp.array(ensemble_observations), axis=0)

        g2_ideal_minus1 = g2_ideal - baseline
        clean_obs_list.append(g2_ideal_minus1)

        snr = gen_beta / jnp.std((noisy_g2_minus_1 - g2_ideal_minus1))
        snrs.append(snr)

        r = g2_ideal_minus1 - noisy_g2_minus_1
        rmse = jnp.sqrt(jnp.sum(jnp.square(r)) / g2_ideal_minus1.size)
        noise_errors.append(rmse)

        noisy_obs_list.append(noisy_g2_minus_1)
        # noisy_obs_list.append(g2_ideal - baseline)

    return clean_obs_list, noisy_obs_list, noise_errors, snrs


# average_counts_khz = 1500
gen_beta = 0.7

ensemble_size = 1

clean_obs_list, noisy_obs_list, noise_errors, snrs = gen_data(
    q,
    t,
    valid_params,
    ensemble_size,
    gen_beta=gen_beta,
    baseline=1.0,
    rand_key=jax.random.key(1337),
    count_rate_khz=20.0, # average count rate for HeNe laser
    experiment_measurement_time_s=30.0, # total measurement time in seconds
    noise_scaling_factor=0.1
)

print(f"error due to noise (avg rmse): {jnp.average(jnp.array(noise_errors))}")
print(f"average SNR: {jnp.average(jnp.array(snrs))}")



error due to noise (avg rmse): 0.004917107035232236
average SNR: 142.56733465299493


In [13]:
clean_obs_list[10]

Array([[0.69995019, 0.69992528, 0.69990038, ..., 0.        , 0.        ,
        0.        ],
       [0.69993276, 0.69989914, 0.69986552, ..., 0.        , 0.        ,
        0.        ],
       [0.69991301, 0.69986953, 0.69982604, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.69934365, 0.69901571, 0.69868792, ..., 0.        , 0.        ,
        0.        ],
       [0.69932392, 0.69898613, 0.6986485 , ..., 0.        , 0.        ,
        0.        ],
       [0.69930651, 0.69896002, 0.69861371, ..., 0.        , 0.        ,
        0.        ]], dtype=float64)

In [ ]:
# visualize a couple noisy observations

# viz_num_pairs = 4
viz_num_pairs = len(noisy_obs_list)

plt.clf()
fig, axs = plt.subplots(viz_num_pairs, 2, figsize=(12, 3*viz_num_pairs), dpi=150)

xx = jnp.linspace(0, 3e-5, 300)
for i in range(viz_num_pairs):
    amp, mu, sig = valid_params[i]
    srcs = normal_distribution(xx, amp, mu, sig)
    obs = clean_obs_list[i] - noisy_obs_list[i]
    # draw each side by side
    # axs[i, 0].set_xscale('log')
    axs[i, 1].set_xscale('log')
    axs[i, 0].plot(xx, srcs)
    axs[i, 1].plot(t, obs.T)
    axs[i, 0].set_title(f"observations")
    axs[i, 1].set_title(f"srcs: {i+1} ({mu[0]:.2e}, {mu[1]:.2e})")
    axs[i, 0].set_xlabel("Diffusion coefficient (m^2/s)")
    axs[i, 1].set_xlabel("Time (s)")
    # axs[i, 0].set_ylabel("Probability density")
    axs[i, 1].set_ylabel("g2(t)")

fig.tight_layout()
plt.show()

In [ ]:
def rilt_observation_matrix(q, t, possible_D, x):
    A = jnp.exp(jnp.einsum('i,j,k->ijk', -possible_D, q**2, t))
    return jnp.einsum('i,ijk->jk', x, A)

def zero_at_ends_rilt(q, t, possible_D, x):
    x_ = x.at[0].set(0.0).at[-1].set(0.0)
    return rilt_observation_matrix(q, t, possible_D, x_)

def dummy_bounds(k):
    return (jnp.zeros(k),), (1e1 * jnp.ones(k),)


def L1_norm(x, alpha):
    return alpha * jnp.sum(jnp.abs(x))

L1_grad = jax.grad(L1_norm)
L1_hess = jax.hessian(L1_norm)

def L1_regularizer(x, alpha=1.0):
    return L1_norm(x, alpha), L1_grad(x, alpha), L1_hess(x, alpha)


def L2_norm(x, alpha):
    return alpha * jnp.sqrt(jnp.sum(jnp.square(x)))

L2_grad = jax.grad(L2_norm)
L2_hess = jax.hessian(L2_norm)

def L2_regularizer(x, alpha=1.0):
    return L2_norm(x, alpha), L2_grad(x, alpha), L2_hess(x, alpha)

# possible_D = jnp.logspace(-6.5, -4.5, 30) * SCALING_CONST
possible_D = jnp.linspace(1e-7, 3e-5, 40) * SCALING_CONST

# contin_opt = HNMFOptimizer(
contin_opt = NewHNMFOptimizer(
# contin_opt = PerturbanceHNMFOptimizer(
# contin_opt = RedoHNMFOptimizer(
    model_fn=zero_at_ends_rilt,
    param_generator=InitParamsGenerator2(dummy_bounds),
    bound_generator=dummy_bounds,
    input_args = ('q', 't', 'possible_D'),
    param_args=('x'),
    constants = {},
    min_k=len(possible_D),
    max_k=len(possible_D),
    nsim=5,
    regularizer_fn=functools.partial(L2_regularizer, alpha=0.005)
    # regularizer_fn=functools.partial(L1_regularizer, alpha=1.0)
)



In [ ]:
rilt_sols_list = []


In [ ]:
rand_key = jax.random.key(808)
for i in range(len(rilt_sols_list), len(noisy_obs_list)):
    # if i > 3:
    #     break
    all_res = contin_opt((q, t, possible_D), noisy_obs_list[i], opt_options={
        'fatol': 1e-14,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-12
    })
    sols = jnp.stack(all_res.sort_values('fval')['sol'].apply(lambda l: l[0]).tolist())
    sols = sols.at[:, 0].set(0).at[:, -1].set(0)
    rilt_sols_list.append(sols)
    print(f"Pair {i+1} done\n\n")



# og_res = rilt_sols_list[0]
new_res = rilt_sols_list[0]


In [ ]:
rand_key = jax.random.key(811)
for i in range(len(rilt_sols_list), len(noisy_obs_list)):
    if i > 3:
        break
    perturbed_obs_list = []
    for j in range(contin_opt.nsim):
        rand_key, subkey = jax.random.split(rand_key)
        perturbed_obs = jax.random.normal(subkey, shape=noisy_obs_list[i].shape) * gen_beta * 0.00000001 + noisy_obs_list[i]
        perturbed_obs_list.append(perturbed_obs)

    all_res = contin_opt((q, t, possible_D), perturbed_obs_list, opt_options={
        'fatol': 1e-14,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-12
    })
    sols = jnp.stack(all_res.sort_values('fval')['sol'].apply(lambda l: l[0]).tolist())
    sols = sols.at[:, 0].set(0).at[:, -1].set(0)
    rilt_sols_list.append(sols)
    print(f"Pair {i+1} done\n\n")
perturbed_res = rilt_sols_list[0]


In [9]:
# set up the multi-phase optimization

optimizer_dirac_single = NewHNMFOptimizer(
# optimizer_dirac_single = PerturbanceHNMFOptimizer(
    model_fn=g2_minus1_matrix_dirac,
    param_generator=InitParamsGenerator2(gen_bounds_dirac),
    bound_generator=gen_bounds_dirac,
    input_args = ('q', 't'),
    param_args=('D', 'amp', 'beta'),
    constants = {"const": SCALING_CONST},
    min_k=min_k,
    max_k=max_k,
    nsim=100
)


def gen_bounds_normal_std(num_sources):
    lower_bounds = (
        # 1e-9*jnp.ones(num_sources),
        # 1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        # jnp.array([0.0])
    )
    upper_bounds = (
        # jnp.inf*jnp.ones(num_sources),
        # 1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        # jnp.array([1.0])
    )
    return lower_bounds, upper_bounds

std_opts = {}
for k in range(min_k, max_k + 1):
    std_opt = NewHNMFOptimizer(
        model_fn=g2_minus1_matrix,
        param_generator=InitParamsGenerator2(gen_bounds_normal_std),
        bound_generator=gen_bounds_normal_std,
        input_args = ('q', 't', 'amp', 'mu', 'beta'),
        param_args=('sig',),
        constants = {},
        min_k=k,
        max_k=k,
        nsim=20
    )
    std_opts[k] = (std_opt)


def gen_bounds_normal_final(num_sources):
    lower_bounds = (
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        1e-9*jnp.ones(num_sources),
        jnp.array([0.0])
    )
    upper_bounds = (
        jnp.inf*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        1e-3*jnp.ones(num_sources),
        jnp.array([1.0])
    )
    return lower_bounds, upper_bounds

final_opt = NewHNMFOptimizer(
# final_opt = PerturbanceHNMFOptimizer(
    # model_fn=g1_matrix,
    model_fn=g2_minus1_matrix,
    param_generator=InitParamsGenerator2(gen_bounds_normal_final),
    bound_generator=gen_bounds_normal_final,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig', 'beta'),
    constants = {},
    min_k=min_k,
    max_k=max_k,
    nsim=100
)


inputs = (q, t)
opt_options = {
    'fatol': 1e-14,
    'frtol': 0,
    'maxiter': 2000,
    'gatol': 1e-12
}



In [10]:
opt_options

{'fatol': 1e-14, 'frtol': 0, 'maxiter': 2000, 'gatol': 1e-12}

In [10]:
rand_key = jax.random.key(495)

final_full_sols = []
final_clust_sols = []

# quad_final_full_sols = []
# quad_final_clust_sols = []

In [13]:
class InitParamFeeder:
    def __init__(self, params_list_map):
        self.params_list_map = params_list_map
        self.counter = {k: 0 for k in params_list_map.keys()}

    def __call__(self, num_sources):
        params = self.params_list_map[num_sources][self.counter[num_sources]]
        self.counter[num_sources] += 1
        return params

for i in range(len(final_clust_sols), len(noisy_obs_list)):
    # observations = noisy_obs_list[i]
    observations = clean_obs_list[i]


    # phase 1 - fit dirac model, gives centers and amplitudes
    g2_res = optimizer_dirac_single((q, t), observations, opt_options=opt_options)
    clust_sol = process_res_dirac(g2_res)

    print("finished phase 1 for pair", i)


    # dirac_sols.append((D, amp, beta))

    # phase 2 - convert dirac to normal, only fit the standard deviation of the gaussians
    # centers and amplitudes are taken from the first phase and fixed

    params_to_feed = {}
    for k in clust_sol.index:
        D, amp, betas = clust_sol.loc[k]['centers']
        if not isinstance(D, jnp.ndarray):
            D = jnp.array(D, ndmin=1)
        if not isinstance(amp, jnp.ndarray):
            amp = jnp.array(amp, ndmin=1)
        if not isinstance(betas, jnp.ndarray):
            betas = jnp.array(betas, ndmin=1)
        beta = betas[0:1]
        std_sols = std_opts[k]((q, t, amp, D, beta), observations, opt_options=opt_options)
        sig_sol = process_res_std_g1(std_sols)['centers'].iloc[0][0]
        if not isinstance(sig_sol, jnp.ndarray):
            sig_sol = jnp.array(sig_sol, ndmin=1)

        nsim = 100
        params_to_feed[k] = []
        for j in range(nsim):
            rand_key, subkey1, subkey2, subkey3 = jax.random.split(rand_key, 4)
            D_ = D + jax.random.normal(subkey1, D.shape) * 0.05 * D
            sig_ = sig_sol + jax.random.normal(subkey2, sig_sol.shape) * 0.05 * sig_sol
            amp_ = amp + jax.random.normal(subkey3, amp.shape) * 0.05 * amp
            params_to_feed[k].append((amp_, D_, sig_, beta))

    print("finished phase 2 for pair", i)

    # final phase - fit with all parameters free

    final_opt.reset_param_generator(InitParamFeeder(params_to_feed))

    final_g2_sols = final_opt((q, t), observations, opt_options=opt_options)
    final_full_sols.append(final_g2_sols)
    final_clust_sol = process_res_normal(final_g2_sols)
    final_clust_sols.append(final_clust_sol)

    with open('distance_clust_sols.pkl', 'wb') as f:
        pickle.dump(final_clust_sols, f)

    print(f"(noisy) Pair {i} done\n\n")



SIMULATIONS FOR 1 SOURCES TOOK 2.93430757522583 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 5.466465473175049 SECONDS
Clustering  2
finished phase 1 for pair 23
SIMULATIONS FOR 1 SOURCES TOOK 3.235459089279175 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 21.911576509475708 SECONDS
Clustering  2
finished phase 2 for pair 23
SIMULATIONS FOR 1 SOURCES TOOK 0.7307610511779785 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 57.05280089378357 SECONDS
Clustering  2
(noisy) Pair 23 done


SIMULATIONS FOR 1 SOURCES TOOK 2.519146203994751 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 2.489603042602539 SECONDS
Clustering  2
finished phase 1 for pair 24
SIMULATIONS FOR 1 SOURCES TOOK 3.3904683589935303 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 22.156009912490845 SECONDS
Clustering  2
finished phase 2 for pair 24
SIMULATIONS FOR 1 SOURCES TOOK 1.325148344039917 SECONDS
SIMULATIONS FOR 2 SOURCES TOOK 17.91090488433838 SECONDS
Clustering  2
(noisy) Pair 24 done


SIMULATIONS FOR 1 SOURCES TOOK 2.3232078552246094 SECONDS
SIMULATIO

In [ ]:
### save solutions with pickle ###

# with open('distance_sols.pkl', 'wb') as f:
#     ress = (final_full_sols, final_clust_sols)
#     pickle.dump(ress, f)


# save final_full_sols with pickle
# with open('quad_final_full_sols.pkl', 'wb') as f:
#     pickle.dump(quad_final_full_sols, f)


In [22]:
with open('distance_clust_sols.pkl', 'rb') as f:
    final_clust_sols = pickle.load(f)


In [26]:
final_clust_sols[0]

,aic_score,avg_sillhouette_score,min_sillhouette_score,reconstruction_loss,centers
num_source,,,,,
1,-13.2505745165997,1.000000,1.000000,0.000488,"[1.0107029599297888, 2.049691752798551e-06, 6...."
2,-21.446198725847545,0.909805,0.857226,0.000003,"[[0.633646703281309, 0.3716540971605493], [2.1..."


In [28]:
num_resids = noisy_obs_list[0].size


def AIC3(sill_avg, recon, num_sources, sill_cutoff=0.7):
    if sill_avg > sill_cutoff:
        aic = 2*num_sources + 2*jnp.log(recon)
    else:
        aic = jnp.inf
    return aic

def BIC(sill_avg, recon, num_resids, num_sources, sill_cutoff=0.7):
    if sill_avg > sill_cutoff:
        aic = num_sources*jnp.log(num_resids) + 2*jnp.log(recon)
    else:
        aic = jnp.inf
    return aic

# for df in final_clust_sols:
#     df['aic_score'] = df.apply(lambda row: AIC3(row['min_sillhouette_score'], row['reconstruction_loss'], row.name), axis=1)

for df in final_clust_sols:
    df['aic_score'] = df.apply(lambda row: BIC(row['min_sillhouette_score'], row['reconstruction_loss'], num_resids, row.name), axis=1)

# final_clust_sols[0].apply(lambda row: BIC(row['min_sillhouette_score'], row['reconstruction_loss'], num_resids, row.name), axis=1)
# final_clust_sols[0].apply(lambda row: row.name, axis=1)


In [33]:
for i, df in enumerate(final_clust_sols):
    print(i, df[['aic_score', 'min_sillhouette_score']])


0                      aic_score  min_sillhouette_score
num_source                                           
1           -6.925753217830918               1.000000
2            -8.79655612830998               0.857226
1                      aic_score  min_sillhouette_score
num_source                                           
1            2.530191371668037               1.000000
2           1.5098370839897122               0.998519
2                       aic_score  min_sillhouette_score
num_source                                            
1            5.0270125213345285               1.000000
2           -25.507968182591256               0.999857
3                      aic_score  min_sillhouette_score
num_source                                           
1            6.189164900998971               1.000000
2           -31.22245056511882               0.999784
4                       aic_score  min_sillhouette_score
num_source                                            
1           

In [29]:
##################################
### run for ensemble_size = 5  ###
###       new noise method     ###
##################################

best_sols_matlab = loadmat("best_sols.mat")['best_sols']
best_sols_matlab = best_sols_matlab.reshape(best_sols_matlab.shape[0], 2, 3).swapaxes(1, 2)


xx = jnp.linspace(0, 4e-5, 300)

# num_rows = min(len(rilt_sols_list), len(final_sols))
num_rows = len(final_clust_sols)
# num_rows = 2
plt.clf()
# fig, axs = plt.subplots(len(final_sols), 1, figsize=(16, 7*len(final_sols)), dpi=150)
fig, axs = plt.subplots(num_rows, 1, figsize=(16, 9*num_rows), dpi=150)

for i in range(num_rows):
    observations = clean_obs_list[i]


    ####### use l-statistic or aic info? #######
    ind = final_clust_sols[i]['aic_score'].argmin()
    n_srcs = final_clust_sols[i].iloc[ind].name
    # n_srcs = 1

    # amp_fin, D_fin, sig_fin, beta_fin = final_full_sols[i][final_full_sols[i]['num_sources'] == n_srcs].sort_values('fval')['sol'].iloc[0]

    # n_srcs, _, _ = l_statistic2(final_full_sols[i], final_clust_sols[i], noisy_obs_list[i], q, t, sill_threshold=0.6, p_threshold=0.05)

    # n_srcs, _, _ = l_statistic(final_full_sols[i], final_clust_sols[i])
    # n_srcs, _, _ = l_statistic(quad_final_full_sols[i], quad_final_clust_sols[i])


    # ind = final_clust_sols[i]['aic_score'].argmin()
    # n_srcs = final_clust_sols[i].iloc[ind].name

    ############################################


    amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    beta_fin = 0.7


    if not isinstance(amp_fin, jnp.ndarray):
        amp_fin = jnp.array(amp_fin, ndmin=1)
    if not isinstance(D_fin, jnp.ndarray):
        D_fin = jnp.array(D_fin, ndmin=1)
    if not isinstance(sig_fin, jnp.ndarray):
        sig_fin = jnp.array(sig_fin, ndmin=1)


    # pred_srcs = normal_distribution(xx, amp, D, sig_sol)
    true_amp, true_mu, true_std = valid_params[i]
    true_srcs = normal_distribution(xx, true_amp, true_mu, true_std).T
    pred_srcs_final = normal_distribution(xx, amp_fin, D_fin, sig_fin).T

    clean_g2minus1 = g2_minus1_matrix(q, t, true_amp, true_mu, true_std, gen_beta)
    r = clean_g2minus1 - observations
    noise_level = 0.5 * jnp.sum(jnp.square(r))
    r = g2_minus1_matrix(q, t, amp_fin, D_fin, sig_fin, beta_fin) - observations
    fit_loss = 0.5*jnp.sum(jnp.square(r))

    # r = rilt_observation_matrix(q, t, possible_D, rilt_sols_list[i][0]) - observations
    # contin_loss = 0.5*jnp.sum(jnp.square(r))

    xx_ = xx * SCALING_CONST

    ax = axs[i]
    ax.plot(xx_, true_srcs, color='yellow', label='True distribution', linestyle=':')
    # ax.vlines(D, 0, amp*0.1, color='red', alpha=0.7, linestyle='-', label='phase 1 fit (dirac)')
    # ax.plot(xx, pred_srcs, color='teal', linestyle='-', label='phase 2 fit (width)')
    ax.plot(xx_, pred_srcs_final, color='green', linestyle='-', label='predicted distrubution (our model)')

    # if i < len(best_sols_matlab):
    #     amp_matlab, D_matlab, sig_matlab = best_sols_matlab[i]
    #     pred_srcs_matlab = normal_distribution(xx, amp_matlab, D_matlab, sig_matlab)
    #     ax.plot(xx_, pred_srcs_matlab, color='turquoise', linestyle='-.', label='matlab code prediction')


    kl_our_model = jnp.sum(jax.scipy.special.rel_entr(true_srcs, pred_srcs_final))

    # rilt_eval_points = possible_D/SCALING_CONST
    # for j in range(len(rilt_sols_list[i])):
    #     rilt_sol = rilt_sols_list[i][j]
    #     rilt_sols_xx = jnp.interp(xx, rilt_eval_points, rilt_sol/10)
    #     if j == 0:
    #         label = 'CONTIN solutions (scaled by .1)'
    #         true_srcs_eval = normal_distribution(rilt_eval_points, true_amp, true_mu, true_std)
    #         kl_contin = jnp.sum(jax.scipy.special.rel_entr(true_srcs_eval, (rilt_sol + jnp.finfo(float).eps))/jnp.sum(rilt_sol))
    #     else:
    #         label = None
    #     ax.plot(xx_, rilt_sols_xx, alpha = 0.5, color='turquoise', linestyle='-.', label=label)



        # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='--', label=f'CONTIN solution {j+1}')
    # rilt_sol = rilt_sols_list[i][0] # top result
    # rilt_sols_xx = jnp.interp(xx, possible_D/SCALING_CONST, rilt_sol/10)
    # ax.plot(xx, rilt_sols_xx, linestyle='-.', label=f'best CONTIN solution (scaled by .1)')
    # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='-.', label=f'best CONTIN solution')

    ax.set_xlabel('Diffusion Coefficient (m^2/s)')

    ax.legend()
    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, (best CONTIN fit): {kl_contin:.2e}")

    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}")
    ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, num srcs: {n_srcs}")

plt.show()


<Figure size 1440x810 with 0 Axes>

In [ ]:
import numpy as np
import miepython
import matplotlib.pyplot as plt

# --- 1. Define Input Parameters ---
m_particle = 1.59      # Refractive index of particle (e.g., Polystyrene)
n_medium = 1.33        # Refractive index of medium (e.g., Water)
wavelength_nm = 633    # Wavelength of the laser in nm
theta_deg = 173        # Scattering angle in degrees

# --- 2. Prepare Inputs for miepython ---
# The complex refractive index m = n_particle / n_medium
# Assuming non-absorbing particles, so the imaginary part is 0.
m = m_particle / n_medium

# Define a range of particle diameters to analyze
diameters_nm = np.linspace(10, 1000, 700)
radii_nm = diameters_nm / 2

# Calculate the dimensionless size parameter 'x'
x = 2 * np.pi * radii_nm * n_medium / wavelength_nm

# Calculate mu = cos(theta)
# The function requires the angle's cosine, not the angle itself.
mu = np.cos(np.deg2rad(theta_deg))

# --- 3. Calculate Mie Scattering Intensity ---
# This is the correct function call based on the docstring
intensity = []
for x_ in x:
    # Calculate the unpolarized intensity for each size parameter
    # The function returns the intensity normalized by the geometric cross-section
    # intensity.append(miepython.i_unpolarized(m, x_, mu))
    miepython.intensities(m, x_, wavelength_nm, mu)
# intensity = miepython.i_unpolarized(m, x, mu)

# --- 4. Plot the Results ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(diameters_nm, intensity, lw=2)
ax.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax.set_ylabel("Normalized Unpolarized Intensity (a.u.)", fontsize=12)
ax.set_title(f"Mie Scattering Intensity at {theta_deg}° for Polystyrene in Water", fontsize=14)
# ax.set_yscale('log')
ax.grid(True, which="both", ls="--")

plt.show()

In [ ]:
import numpy as np
import miepython
import matplotlib.pyplot as plt

# --- 1. Define Input Parameters ---
m_particle = 1.59      # Refractive index of particle (Polystyrene)
n_medium = 1.33        # Refractive index of medium (Water)
wavelength_nm = 633    # Wavelength of the laser in nm
theta_deg = 173        # Scattering angle in degrees

# --- 2. Prepare Inputs for miepython ---
m = m_particle / n_medium
diameters_nm = np.linspace(0.1, 1000, 700)
radii_nm = diameters_nm / 2
mu = np.cos(np.deg2rad(theta_deg))

# --- 3. Calculate Scattering Properties (using a loop) ---
# Initialize lists to store the results
unnorm_intensity_list = []
C_sca_list = []

# Loop over each particle size
for r in radii_nm:
    # Calculate size parameter for the current radius
    x_val = 2 * np.pi * r * n_medium / wavelength_nm

    # Calculate scattering efficiencies and intensity for this size
    # Note: mie() expects a scalar x, so we call it inside the loop
    # qext, qsca, qback, g = miepython.mie(m, x_val)
    norm_intensity = miepython.i_unpolarized(m, x_val, mu)
    
    # Calculate the un-normalized intensity (∝ d⁶)
    unnorm_intensity_list.append(norm_intensity * x_val**2)
    
    # Calculate the scattering cross-section (nm^2)
    # C_sca_list.append(qsca * np.pi * r**2)

# Convert lists to numpy arrays for plotting
unnorm_intensity = np.array(unnorm_intensity_list)
C_sca = np.array(C_sca_list)

# --- 4. Plot the Results ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Corrected Plots Showing Expected Rayleigh Behavior", fontsize=16)

# Plot un-normalized intensity
ax1.plot(diameters_nm, unnorm_intensity, lw=2, color='crimson')
ax1.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax1.set_ylabel("Un-normalized Intensity (a.u.)", fontsize=12)
ax1.set_title(f"Un-normalized Intensity at {theta_deg}°", fontsize=14)
ax1.grid(True, which="both", ls="--")

# Plot scattering cross-section
# ax2.plot(diameters_nm, C_sca, lw=2, color='darkgreen')
# ax2.set_xlabel("Particle Diameter (nm)", fontsize=12)
# ax2.set_ylabel("Scattering Cross-Section C_sca (nm²)", fontsize=12)
# ax2.set_title("Total Scattering Cross-Section", fontsize=14)
# ax2.grid(True, which="both", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import numpy as np
import miepython as mie
import matplotlib.pyplot as plt

# --- 1. Define Core Physical Parameters ---
m_particle = 1.59      # Refractive index of particle (Polystyrene)
n_medium = 1.33        # Refractive index of medium (Water)
wavelength_nm = 633    # Wavelength of the laser in nm
theta_deg = 173        # Scattering angle in degrees

# Define the range of particle diameters
diameters_nm = np.linspace(0.1, 1000, 700)

# --- 2. Prepare Inputs and Initialize Lists ---
# Calculate constant values outside the loop
m_relative = m_particle / n_medium
mu = np.cos(np.deg2rad(theta_deg))

# Initialize empty lists to store the results from each iteration
I_par_list = []
I_per_list = []
C_sca_list = []

# --- 3. Calculate Scattering Properties with a Loop ---
# We must loop because the library functions expect a single particle size
for d in diameters_nm:
    # Calculate radius and size parameter 'x' for the current diameter
    r = d / 2
    x = 2 * np.pi * r * n_medium / wavelength_nm
    
    # Calculate scattering amplitudes S1 and S2 for this particle
    s1, s2 = mie.S1_S2(m_relative, x, mu)
    
    # The intensities are the magnitude squared of the amplitudes
    I_par_list.append(np.abs(s1)**2)
    I_per_list.append(np.abs(s2)**2)
    
    # Calculate scattering efficiency to find the total cross-section
    qext, qsca, qback, g = mie.efficiencies_mx(m_relative, x)
    C_sca_list.append(qsca * np.pi * r**2)

# Convert the lists of results into NumPy arrays for plotting
I_par = np.array(I_par_list)
I_per = np.array(I_per_list)
C_sca = np.array(C_sca_list)

# --- 4. Plot the Results ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Final Working Mie Scattering Plots", fontsize=16)

# Plot polarized intensities
ax1.plot(diameters_nm, I_per, lw=2, label='Perpendicular Pol. ($I_{\perp} = |S_2|^2$)')
ax1.plot(diameters_nm, I_par, lw=2, label='Parallel Pol. ($I_{\parallel} = |S_1|^2$)', ls='--')
ax1.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax1.set_ylabel("Un-normalized Intensity (a.u.)", fontsize=12)
ax1.set_title(f"Polarized Intensity at {theta_deg}°", fontsize=14)
ax1.legend()
ax1.grid(True, which="both", ls="--")

# Plot scattering cross-section
ax2.plot(diameters_nm, C_sca, lw=2, color='darkgreen')
ax2.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax2.set_ylabel("Scattering Cross-Section $C_{sca}$ (nm²)", fontsize=12)
ax2.set_title("Total Scattering (shows $d^6$ trend)", fontsize=14)
ax2.grid(True, which="both", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import numpy as np
import miepython as mie
import matplotlib.pyplot as plt

def convert_volume_to_intensity_rayleigh(diameters_nm, volume_dist):
    """
    Converts a volume-weighted PSD to an intensity-weighted PSD using
    the Rayleigh approximation (I ~ d^6).

    Args:
        diameters_nm (np.ndarray): Array of particle diameters in nm.
        volume_dist (np.ndarray): Corresponding volume-weighted distribution values.

    Returns:
        np.ndarray: The normalized intensity-weighted distribution.
    """
    # Weighting factor is d^6 / d^3 = d^3
    weighting_factor = diameters_nm**3
    
    # Apply the weighting factor
    unnormalized_intensity_dist = volume_dist * weighting_factor
    
    # Normalize the final distribution to sum to 1
    total_intensity = np.sum(unnormalized_intensity_dist)
    if total_intensity == 0:
        return unnormalized_intensity_dist
        
    return unnormalized_intensity_dist / total_intensity

def convert_volume_to_intensity_mie(
    diameters_nm, 
    volume_dist, 
    m_particle, 
    n_medium, 
    wavelength_nm, 
    theta_deg
):
    """
    Converts a volume-weighted PSD to an intensity-weighted PSD using
    full Mie theory for a specific angle (ideal for DLS).
    """
    # Calculate constant values
    m_relative = m_particle / n_medium
    mu = np.cos(np.deg2rad(theta_deg))
    
    # Calculate the angle-specific Mie scattering intensity for each diameter
    mie_intensity_at_angle = []
    for d in diameters_nm:
        if d == 0:
            mie_intensity_at_angle.append(0)
            continue
        r = d / 2
        x = 2 * np.pi * r * n_medium / wavelength_nm
        s1, s2 = mie.S1_S2(m_relative, x, mu)
        
        # Intensity for unpolarized light is the average of the two polarizations
        intensity = 0.5 * (np.abs(s1)**2 + np.abs(s2)**2)
        mie_intensity_at_angle.append(intensity)
    
    mie_intensity_at_angle = np.array(mie_intensity_at_angle).flatten()
    
    # --- THE FIX IS HERE ---
    # The weighting factor should MULTIPLY by d^3, not divide.
    weighting_factor = mie_intensity_at_angle * (diameters_nm**3)

    # Apply the weighting factor
    unnormalized_intensity_dist = volume_dist * weighting_factor
    
    # Normalize the final distribution to sum to 1
    total_intensity = np.sum(unnormalized_intensity_dist)
    if total_intensity == 0:
        return unnormalized_intensity_dist
        
    return unnormalized_intensity_dist / total_intensity

In [ ]:
# volume_dist.shape

np.ones_like(volume_dist) / volume_dist.shape

In [ ]:
# --- 1. Define Physical Parameters for Mie Calculation ---
m_particle = 1.59      # Polystyrene
n_medium = 1.33        # Water
wavelength_nm = 633    # HeNe laser
theta_deg = 150        # DLS backscatter angle

# --- 2. Create a Sample Volume-Weighted PSD ---
# This simulates a bimodal distribution with peaks at 80 nm and 400 nm
diameters_nm = np.linspace(1, 1000, 500)
peak1 = 0.7 * np.exp(-((diameters_nm - 80)**2) / (2 * 20**2))
peak2 = 0.3 * np.exp(-((diameters_nm - 700)**2) / (2 * 50**2))
volume_dist = peak1 + peak2

volume_dist = np.ones_like(volume_dist)

# Normalize the starting distribution
volume_dist /= np.sum(volume_dist)

# --- 3. Perform the Conversions ---
intensity_dist_rayleigh = convert_volume_to_intensity_rayleigh(diameters_nm, volume_dist)
intensity_dist_mie = convert_volume_to_intensity_mie(
    diameters_nm, 
    volume_dist, 
    m_particle, 
    n_medium, 
    wavelength_nm, 
    theta_deg
)

# --- 4. Plot the Results for Comparison ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(diameters_nm, volume_dist * 100, lw=2.5, color='black', label='Original Volume-Weighted PSD')
ax.plot(diameters_nm, intensity_dist_rayleigh * 100, lw=2, ls='--', color='dodgerblue', label='Intensity (Rayleigh approx.)')
# ax.plot(diameters_nm, intensity_dist_mie * 100, lw=2, ls='-', color='crimson', label=f'Intensity (Mie, {theta_deg}°)')
ax.plot(diameters_nm, intensity_dist_mie * 100, lw=2, ls='-', color='crimson', label=f'Intensity (Mie, {theta_deg}°)')

ax.set_xlabel("Particle Diameter (nm)", fontsize=12)
ax.set_ylabel("Relative Contribution (%)", fontsize=12)
ax.set_title("PSD Conversion: Volume to Intensity", fontsize=16)
ax.legend()
ax.grid(True, which="both", ls="--")
ax.set_xlim(0, 1000)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

In [ ]:
intensity_dist_rayleigh.shape

In [ ]:
intensity_dist_mie.shape

In [ ]:
import numpy as np
import miepython
import matplotlib.pyplot as plt

# --- Parameters (same as before) ---
m = 1.59 / 1.33
wavelength_nm = 633
theta_deg = 173
mu = np.cos(np.deg2rad(theta_deg))

# --- Data Generation (same as before) ---
diameters_nm = np.linspace(10, 1000, 500)
radii_nm = diameters_nm / 2
x = 2 * np.pi * radii_nm * 1.33 / wavelength_nm
# intensity = miepython.i_unpolarized(m, x, mu)
intensity = []
for x_ in x:
    # Calculate the unpolarized intensity for each size parameter
    # The function returns the intensity normalized by the geometric cross-section
    intensity.append(miepython.i_unpolarized(m, x_, mu))


# --- Create new plots to inspect the Rayleigh region ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Inspecting the 'Flat' Region (d < 100 nm)", fontsize=16)

# Plot 1: The original Log-Lin view (zoomed in)
ax1.plot(diameters_nm, intensity)
# ax1.set_yscale('log')
ax1.set_xlim(0, 100)
ax1.set_ylim(1e-5, 0.2) # Adjust ylim to see the curve
ax1.set_title("Original Log-Lin View")
ax1.set_xlabel("Particle Diameter (nm)")
ax1.set_ylabel("Normalized Intensity")
ax1.grid(True, which="both", ls="--")

# Plot 2: A Log-Log view to reveal the power law
ax2.plot(diameters_nm, intensity)
ax2.set_xscale('log')
# ax2.set_yscale('log')
ax2.set_xlim(10, 100)
ax2.set_ylim(1e-5, 0.2)
ax2.set_title("Log-Log View Reveals the Power Law")
ax2.set_xlabel("Particle Diameter (nm)")
ax2.set_ylabel("Normalized Intensity")
ax2.grid(True, which="both", ls="--")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
def _single_bspline_eval(x, control_points, knots, degree):
    """Helper function to evaluate a B-spline for a single scalar x."""
    n = control_points.shape[0]
    
    # Initialize basis functions for degree 0
    # N_{i,0}(x) = 1 if knots[i] <= x < knots[i+1], else 0
    basis_funcs = (x >= knots[:n]) & (x < knots[1:n+1])
    
    # Recursively compute basis functions for higher degrees
    for d in range(1, degree + 1):
        # Temporarily store the basis functions from the previous degree
        prev_basis = basis_funcs
        # We need n basis functions of degree d, so we size the array for it
        basis_funcs = jnp.zeros(n)
        
        # --- First Term of the recursion ---
        # Denominator: knots[i+d] - knots[i]
        denom1 = knots[d:d+n] - knots[:n]
        # Handle division by zero for coincident knots
        term1 = jnp.where(
            denom1 > 0,
            (x - knots[:n]) / denom1 * prev_basis,
            0.0
        )
        
        # --- Second Term of the recursion ---
        # Denominator: knots[i+d+1] - knots[i+1]
        denom2 = knots[d+1:d+n+1] - knots[1:n+1]
        # Handle division by zero for coincident knots
        term2 = jnp.where(
            denom2 > 0,
            (knots[d+1:d+n+1] - x) / denom2 * prev_basis,
            0.0
        )
        
        # The new basis functions are the sum of two terms from the previous degree
        # N_{i,d} depends on N_{i,d-1} and N_{i-1,d-1}
        # To align terms correctly for summation:
        # basis_funcs[i] = term1[i] (from N_{i,d-1}) + term2[i-1] (from N_{i-1,d-1})
        basis_funcs = basis_funcs.at[1:].add(term2[:-1])
        basis_funcs = basis_funcs.at[:].add(term1)

    # Final spline value is the dot product of control points and basis functions
    res = jnp.dot(control_points, basis_funcs)
    return jnp.reshape(res, (1, -1))

# bspline = jax.jit(jax.vmap(_single_bspline_eval, in_axes=(0, None, None, None)), static_argnums=(3,))

@functools.partial(jax.jit, static_argnums=(3,))
def bspline(x, control_points, knots, degree):
    return jax.vmap(_single_bspline_eval, in_axes=(0, None, None, None))(x, control_points, knots, degree)



In [ ]:
from scipy.stats import norm

domain_start, domain_end = -4.0, 4.0
x_data = jnp.linspace(domain_start, domain_end, 200)
y_data = jnp.array(norm.pdf(x_data, loc=0, scale=1))

degree = 5
num_control_points = 10

num_total_knots = num_control_points + degree + 1
num_interior_knots = num_total_knots - 2 * degree

knots = jnp.linspace(domain_start, domain_end, num_total_knots)

# def gen_bounds_spline_dummy(k):
#     """Generates dummy bounds for the spline parameters."""
#     # Controls can be any real number, knots are clamped to the domain
#     lower_bounds = (jnp.zeros(num_control_points), domain_start*jnp.ones(num_total_knots))
#     upper_bounds = (jnp.inf*jnp.ones(num_control_points), domain_end*jnp.ones(num_total_knots))
#     return lower_bounds, upper_bounds

def gen_bounds_spline_dummy(k):
    """Generates dummy bounds for the spline parameters."""
    # Controls can be any real number, knots are clamped to the domain
    lower_bounds = (-100.0*jnp.ones(num_control_points), )
    upper_bounds = (jnp.inf*jnp.ones(num_control_points), )
    return lower_bounds, upper_bounds

spline_opt = HNMFOptimizer(
    model_fn=bspline,
    param_generator=InitParamsGenerator2(gen_bounds_spline_dummy),
    bound_generator=gen_bounds_spline_dummy,
    input_args=('x'),
    param_args=('control_points',),
    constants = {'degree': degree, 'knots': knots},
    min_k=1,
    max_k=1,
    nsim=20
)

ress = spline_opt(x_data, y_data, opt_options={
    'fatol': 1e-14,
    'frtol': 0,
    'maxiter': 2000,
    'gatol': 1e-12
})

ress = ress.sort_values('fval')


# final_control, final_knots = ress.iloc[0]['sol']
final_control = ress.iloc[0]['sol'][0]
final_knots = knots  # Use the fixed knots from the optimization
y_final = bspline(x_data, final_control, final_knots, degree)


plt.figure(figsize=(12, 8))
# Plot curves
plt.plot(x_data, jnp.squeeze(y_data), 'r--', label='Target Normal Curve', lw=2)
plt.plot(x_data, jnp.squeeze(y_final), 'b-', label='Fitted B-Spline (Trained)', lw=2.5)

# Plot initial and final knot positions
# plt.scatter(initial_knots, jnp.full_like(initial_knots, -0.02), marker='|', color='gray', s=100, label='Initial Knots')
plt.scatter(final_knots, jnp.full_like(final_knots, -0.04), marker='|', color='purple', s=100, label='Final (Learned) Knots')

plt.title('Fitting a B-Spline with Trainable Knots')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# ress.iloc[0]
final_control

In [ ]:
##################################
### run for ensemble_size = 5  ###
###       new noise method     ###
##################################

best_sols_matlab = loadmat("best_sols.mat")['best_sols']
best_sols_matlab = best_sols_matlab.reshape(best_sols_matlab.shape[0], 2, 3).swapaxes(1, 2)


xx = jnp.linspace(0, 4e-5, 300)

# num_rows = min(len(rilt_sols_list), len(final_sols))
# num_rows = len(final_clust_sols)
num_rows = len(quad_final_clust_sols)

plt.clf()
# fig, axs = plt.subplots(len(final_sols), 1, figsize=(16, 7*len(final_sols)), dpi=150)
fig, axs = plt.subplots(num_rows, 1, figsize=(16, 9*num_rows), dpi=150)

for i in range(num_rows):
    # observations = noisy_obs_list[i]
    observations = clean_obs_list[i]


    ####### use l-statistic or aic info? #######
    # n_srcs, _, _ = l_statistic2(final_full_sols[i], final_clust_sols[i], noisy_obs_list[i], q, t, sill_threshold=0.6, p_threshold=0.05)

    # n_srcs, _, _ = l_statistic(final_full_sols[i], final_clust_sols[i])
    # n_srcs, _, _ = l_statistic(quad_final_full_sols[i], quad_final_clust_sols[i])
    n_srcs = 2


    # ind = final_clust_sols[i]['aic_score'].argmin()
    # n_srcs = final_clust_sols[i].iloc[ind].name

    ############################################


    #### use centroids or use best solution? ###

    # amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    amp_fin, D_fin, sig_fin = final_clust_sols[i].loc[n_srcs]['centers']
    if not isinstance(amp_fin, jnp.ndarray):
        amp_fin = jnp.array(amp_fin, ndmin=1)
    if not isinstance(D_fin, jnp.ndarray):
        D_fin = jnp.array(D_fin, ndmin=1)
    if not isinstance(sig_fin, jnp.ndarray):
        sig_fin = jnp.array(sig_fin, ndmin=1)
    beta_fin = 0.7

    amp_quad, D_quad, sig_quad = quad_final_clust_sols[i].loc[n_srcs]['centers']
    if not isinstance(amp_quad, jnp.ndarray):
        amp_quad = jnp.array(amp_quad, ndmin=1)
    if not isinstance(D_quad, jnp.ndarray):
        D_quad = jnp.array(D_quad, ndmin=1)
    if not isinstance(sig_quad, jnp.ndarray):
        sig_quad = jnp.array(sig_quad, ndmin=1)
    beta_fin = 0.7

    # amp_fin, D_fin, sig_fin, beta_fin = final_full_sols[i][final_full_sols[i]['num_sources'] == n_srcs].sort_values('fval').iloc[0]['sol']

    ############################################

    # pred_srcs = normal_distribution(xx, amp, D, sig_sol)
    true_amp, true_mu, true_std = valid_params[i]
    true_srcs = normal_distribution(xx, true_amp, true_mu, true_std)
    pred_srcs_final = normal_distribution(xx, amp_fin, D_fin, sig_fin)
    pred_srcs_quad = normal_distribution(xx, amp_quad, D_quad, sig_quad)

    clean_g2minus1 = g2_minus1_matrix(q, t, true_amp, true_mu, true_std, gen_beta)
    r = clean_g2minus1 - observations
    noise_level = 0.5 * jnp.sum(jnp.square(r))
    r = g2_minus1_matrix(q, t, amp_fin, D_fin, sig_fin, beta_fin) - observations
    fit_loss = 0.5*jnp.sum(jnp.square(r))

    # r = rilt_observation_matrix(q, t, possible_D, rilt_sols_list[i][0]) - observations
    # contin_loss = 0.5*jnp.sum(jnp.square(r))

    xx_ = xx * SCALING_CONST

    ax = axs[i]
    ax.plot(xx_, true_srcs, color='yellow', label='True distribution', linestyle=':')
    # ax.vlines(D, 0, amp*0.1, color='red', alpha=0.7, linestyle='-', label='phase 1 fit (dirac)')
    # ax.plot(xx, pred_srcs, color='teal', linestyle='-', label='phase 2 fit (width)')
    ax.plot(xx_, pred_srcs_final, color='green', linestyle='-', label='predicted distrubution (our model)')
    ax.plot(xx_, pred_srcs_quad, color='blue', linestyle='-.', label='predicted distrubution (quadrature)', alpha=0.5)

    # if i < len(best_sols_matlab):
    #     amp_matlab, D_matlab, sig_matlab = best_sols_matlab[i]
    #     pred_srcs_matlab = normal_distribution(xx, amp_matlab, D_matlab, sig_matlab)
    #     ax.plot(xx_, pred_srcs_matlab, color='turquoise', linestyle='-.', label='matlab code prediction')


    kl_our_model = jnp.sum(jax.scipy.special.rel_entr(true_srcs, pred_srcs_final))
    # rilt_eval_points = possible_D/SCALING_CONST
    # for j in range(len(rilt_sols_list[i])):
    #     rilt_sol = rilt_sols_list[i][j]
    #     rilt_sols_xx = jnp.interp(xx, rilt_eval_points, rilt_sol/10)
    #     if j == 0:
    #         label = 'CONTIN solutions (scaled by .1)'
    #         true_srcs_eval = normal_distribution(rilt_eval_points, true_amp, true_mu, true_std)
    #         kl_contin = jnp.sum(jax.scipy.special.rel_entr(true_srcs_eval, (rilt_sol + jnp.finfo(float).eps))/jnp.sum(rilt_sol))
    #     else:
    #         label = None
    #     ax.plot(xx_, rilt_sols_xx, alpha = 0.5, color='turquoise', linestyle='-.', label=label)



        # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='--', label=f'CONTIN solution {j+1}')
    # rilt_sol = rilt_sols_list[i][0] # top result
    # rilt_sols_xx = jnp.interp(xx, possible_D/SCALING_CONST, rilt_sol/10)
    # ax.plot(xx, rilt_sols_xx, linestyle='-.', label=f'best CONTIN solution (scaled by .1)')
    # ax.plot(possible_D / SCALING_CONST, rilt_sol, linestyle='-.', label=f'best CONTIN solution')

    ax.set_xlabel('Diffusion Coefficient (m^2/s)')

    ax.legend()
    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, (best CONTIN fit): {kl_contin:.2e}")

    # ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}, loss (best CONTIN): {contin_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}")
    ax.set_title(f"bimodal mix {i} - noise level: {noise_level:.4e}, loss (our model): {fit_loss:.4e}\nKL divergences - (our model): {kl_our_model:.2e}, num srcs: {n_srcs}")

plt.show()
